# 07 — Hierarchical risk parity

`HierarchicalRiskParity`, long-only hierarchical risk parity, against EW and 60/40; hypotheses are pre-registered in §2 and cross-model tests are in notebooks 08–09.

## §1 Hierarchical risk parity

Hierarchical risk parity (HRP; López de Prado 2016) clusters assets by correlation distance and allocates top-down through the resulting tree, splitting capital at each node in inverse proportion to the variance of each side. It never inverts Σ. Correlation distances and minimum-spanning-tree clustering of asset returns go back to Mantegna (1999); single linkage and the minimum spanning tree give the same hierarchy (Gower & Ross 1969). In Monte Carlo experiments López de Prado (2016) reports a lower out-of-sample variance for HRP than for the minimum-variance portfolio. Raffinot (2017) replaces the variance split with equal capital across clusters.

Notebooks 05 and 06 found the UUP corner in MDP and ERC, both of which use the full Σ in the condition that defines their weights. HRP uses only diagonal blocks of Σ. The question is whether UUP then sits in a cluster of its own, and what that does to its weight.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
import sys
import inspect
import textwrap
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import cophenet

import maplab as ml
import _nb07_helpers as h
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, MostDiversified,
    EqualRiskContribution, HierarchicalRiskParity, backtest, summary,
)
from maplab.models import FixedWeight, _min_variance_long_only, _tangency_long_only, _hrp_long_only, _hrp_cluster_var

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
F4 = lambda v: f"{v:.4f}"
E3 = lambda v: f"{v:.3e}"

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")
asof_2022 = pd.Timestamp("2022-12-31")

In [ ]:
ml.inference.render_registration(ml.inference.load_registration("nb07"))

## §3 The math

**General form.** Given a binary tree over the assets, recursive bisection starts from $w_i = 1$ and, at every node with children $L$ and $R$, multiplies the weights on each side by

$$\alpha_L = \frac{V_R}{V_L + V_R}, \qquad \alpha_R = 1 - \alpha_L, \qquad V_S = u_S^\top \Sigma_{SS}\, u_S, \quad u_S \propto \operatorname{diag}(\Sigma_{SS})^{-1},$$

where $u_S$ is the inverse-variance portfolio of side $S$. Each node splits capital between its two sides as the minimum-variance split of two uncorrelated assets with variances $V_L$ and $V_R$. A weight is the product of the shares along its path from the root:

$$w_i = \prod_{\text{nodes on the path to } i} \alpha_{\text{side of } i}.$$

Only the diagonal blocks $\Sigma_{LL}$ and $\Sigma_{RR}$ enter a share; the cross block $\Sigma_{LR}$ only shapes the tree. Nothing is inverted and nothing is optimized.

**The version the class uses.** $C = D^{-1}\Sigma D^{-1}$ (clipped, unit diagonal); $d_{ij} = \sqrt{(1-\rho_{ij})/2}$; single linkage on the Euclidean distance between columns of $d$ (distance of distances); then bisection at each node of the dendrogram (tree-following). The original algorithm orders the leaves and splits the ordered list into halves (positional); its weights depend on the input column order (§4 checks), which is why tree-following is registered.

**Special cases.**

- $C = I$: every share is inverse variance, so HRP = IVP = long-only GMV for any tree. On the same diagonal Σ, MDP and ERC reduce to inverse volatility: HRP is variance-based.
- $N = 2$: inverse variance for any $\rho$.
- $c\Sigma$ leaves the weights unchanged. Per-asset rescaling $K\Sigma K$ keeps the tree but has no simple effect on the weights; there is no analogue of ERC's and MDP's $w(K\Sigma K) \propto K^{-1} w(\Sigma)$.
- Exchangeable Σ (equal vols, constant $\rho > 0$): GMV = MDP = ERC = EW, but positional HRP differs from EW unless $N$ is a power of 2.

**Root-singleton identity.** If asset $i$ is alone on one side of the final merge, its path is the root split alone:

$$w_i = \frac{V_{\text{rest}}}{\sigma_i^2 + V_{\text{rest}}} = \frac{r^2}{1+r^2}, \qquad r = \frac{\sqrt{V_{\text{rest}}}}{\sigma_i},$$

with $V_{\text{rest}}$ the variance of the inverse-variance portfolio of the other $N-1$ assets. It does not depend on asset $i$'s correlations: HRP is correlation-blind for that asset, and $w_i > 1/2$ exactly when $r > 1$.

## §4 The class and the rolling backtest

`HierarchicalRiskParity.predict_weights` estimates Σ with `_estimate` (notebook 01 §3) and passes it to `_hrp_long_only`, which builds the tree and bisects it; `_hrp_cluster_var` is the side variance $V_S$ of §3. There is no solver.

In [ ]:
for obj in (HierarchicalRiskParity.predict_weights, _hrp_long_only, _hrp_cluster_var):
    print(f"# {obj.__module__}.{obj.__qualname__}")
    print(textwrap.dedent(inspect.getsource(obj)))

### Backtests

Ten runs, harness as in notebook 01 §3. HRP(S) is registered; HRP(LW), HRP&#91;pos&#93;(S) and HRP&#91;ward&#93;(S) are the sensitivities and IVP the reference (§2's notebook-local IVP is now the package class `InverseVariance`, same weights); GMV(S), EW, MDP(S), ERC(S) and 60/40 are the reproduction gate, and EW and 60/40 are the benchmarks.

In [ ]:
strategies = {
    "HRP(S)": HierarchicalRiskParity(cov_estimator=ml.sample_cov),
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "ERC(S)": EqualRiskContribution(cov_estimator=ml.sample_cov),
    "HRP(LW)": HierarchicalRiskParity(cov_estimator=ml.ledoit_wolf_cov),
    "HRP[pos](S)": HierarchicalRiskParity(bisection="positional"),
    "HRP[ward](S)": HierarchicalRiskParity(linkage="ward"),
    "IVP": ml.InverseVariance(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    assert wlog.index.equals(rdates), f"{name}: wlog index != rdates"
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

GMV(S), EW, MDP(S), ERC(S) and 60/40 must match notebooks 01, 05 and 06 to 2 dp in every window; otherwise the cell stops the notebook.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4

In [ ]:
ml.inference.check_reproduction(summary_table, {"GMV(S)": "GMV_sample", "EW": "EW", "MDP(S)": "MDP_sample",
                                                "ERC(S)": "ERC_sample", "60/40": "SixtyForty_SPY_IEF"})

print("REPRODUCTION GATE: PASSED -- GMV(S), EW, MDP(S), ERC(S), 60/40 match notebooks 01/05/06 to 2 dp.")

### Arithmetic checks

At all 208 rebalances, under the sample and LW Σ and both bisections, recomputed with `_hrp_long_only`: weights sum to 1 and are positive, every split share matches $V_R/(V_L+V_R)$ from an independent variance function, diagonal Σ gives IVP, scaling Σ and (for tree-following) reordering the columns leave the weights unchanged, and the linkage, leaf order and distance matrix are valid, all to 1e-12; the last lines measure how far positional weights move under 20 random column orders.

In [ ]:
H, pos_spreads_208, cophs, root_sides = h.mechanism_checks(panel, rdates)
print(H.to_string(float_format=E3))

tol = 1e-12
for _, row in H.iterrows():
    assert row["sum_dev"] <= tol
    assert row["min_w"] > 0
    assert row["split_tree"] <= tol
    assert row["split_pos"] <= tol
    assert row["diag_ivp_tree"] <= tol
    assert row["diag_ivp_pos"] <= tol
    assert row["scale_tree"] <= tol
    assert row["scale_pos"] <= tol
    assert row["perm_tree"] <= tol
    assert row["perm_valid"] == 208
    assert row["link_ok"] == 208
    assert row["qd_match"] == 208
    assert row["dist_ok"] == 208

print("Mechanism checks: PASSED at all 208 rebalances, both estimators, both bisections.")

ps = np.array(pos_spreads_208)
n_changes = sum(1 for i in range(1, len(root_sides)) if root_sides[i] != root_sides[i - 1])
print(f"positional permutation spread (K=20, sample Sigma): max={E3(ps.max())}  "
      f"n>1e-12={(ps > 1e-12).sum()}  n>1%={(ps > 0.01).sum()}")
print(f"median cophenetic correlation (sample Sigma, tree): {F4(float(np.median(cophs)))}")
print(f"root-split membership changes across 208 rebalances (HRP(S), tree): {n_changes}")

All checks pass for both estimators and both bisections. Positional weights change with the column order at 182/208 rebalances (by more than 1% half-L1 at 116; largest 12.1%); tree-following weights do not. The median cophenetic correlation is 0.83, and the root split's membership changes 4 times over the 208 rebalances.

## §5 Snapshots

### 5.1 Three dates

HRP, positional HRP, IVP, GMV, MDP, ERC and EW on the sample Σ at the three dates that fixed the H1a, H2b and H3 directions: 2014-12-31, 2017-12-31 and 2022-12-31. Full weights are printed for 2022-12-31 only.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

snap = h.snapshot(panel, DATES)

print("snapshot dates:", ", ".join(str(d.date()) for d in DATES))

In [ ]:
s = snap[asof_2022]
print(f"----- {asof_2022.date()} weights -----")
print(pd.DataFrame(s["W"], index=s["cols"]).to_string(float_format=F4))

Every HRP weight is positive, as §3 requires, but SPY, EFA and VNQ hold under 0.6% each, while UUP (55.30%), GLD (12.60%) and TIP (11.23%) hold 79%. Positional HRP gives UUP 48.92% and IVP 17.86%.

### 5.2 Diversification

Per date: UUP's mean correlation with the other 12 and its volatility rank, then each method's UUP weight, effective N, number of uncorrelated bets, largest variance share, diversification ratio and ex-ante volatility.

In [ ]:
for asof in DATES:
    s = snap[asof]
    Sigma_np, iU, n = s["Sigma_np"], s["iU"], s["n"]
    C = s["C"]
    rho_bar = float((C.sum() - n) / (n * (n - 1)))
    uup_mean_corr = float((C[iU].sum() - 1.0) / (n - 1))
    uup_vol_rank = int((s["sigma_np"] <= s["sigma_np"][iU]).sum())
    print(f"\n----- {asof.date()} -----")
    print(f"UUP mean pairwise corr={F4(uup_mean_corr)}  vol rank (1=lowest)={uup_vol_rank}  rho_bar={F4(rho_bar)}")

    div_tbl = ml.diagnostics.diversification_table(s["W"], Sigma_np, tickers=s["cols"])
    print(div_tbl.to_string(float_format=F4))

At 2022-12-31 HRP holds 55.30% in UUP (GMV 58.12%, MDP 59.22%, ERC 48.36%), with effective N 2.93 and 58.29% of its variance in one asset. UUP's mean correlation with the other 12 is −0.47, and its volatility rank is 1 of 13. HRP's ex-ante volatility, 4.59%, is above GMV's 3.32%, MDP's 3.50% and ERC's 4.32%.

### 5.3 Capital vs risk at 2022-12-31

Bars are capital weights, faded bars variance shares; dashed = 1/13. A negative share means the asset hedges the rest of the portfolio.

In [ ]:
s = snap[asof_2022]
Sigma_2022 = pd.DataFrame(s["Sigma_np"], index=s["cols"], columns=s["cols"])
weights_2022 = {name: pd.Series(s["W"][name], index=s["cols"]) for name in ["HRP", "GMV", "MDP", "ERC", "EW"]}

fig = ml.plotting.capital_vs_risk(weights_2022, Sigma_2022, title="Capital weight vs risk share, 2022-12-31 (sample Σ)")
plt.show()

UUP carries 58.29% of HRP's variance on 55.30% of its capital, and DBC about 15% on 5.96%; EFA and EEM have small negative shares. Unlike ERC, HRP does not equalize variance shares.

### 5.4 The tree and the root-singleton identity

Per date: whether UUP is alone on one side of the final merge, the shares along its path (their product is its weight), its positional path and the leaf order. The next cell checks the §3 identity directly and repeats it under Σ_cf.

In [ ]:
for asof in DATES:
    s = snap[asof]
    Sigma_np, iU, cols = s["Sigma_np"], s["iU"], s["cols"]
    Z = s["info"]["Z"]
    coph = float(cophenet(Z, s["y"])[0])
    print(f"\n----- {asof.date()} -----")
    print(f"UUP root singleton (tree): {h.root_singleton(Z, iU)}   cophenetic corr={F4(coph)}   "
          f"last merge height={F4(Z[-1, 2])}")
    tp = h.tree_path(Sigma_np, Z, iU)
    tp_display = [(a, b, round(c, 4)) for a, b, c in tp]
    print("UUP tree path (own-side n, other-side n, share to own side):", tp_display)
    prod = 1.0
    for _, _, a in tp:
        prod *= a
    print(f"  |path product - w_UUP| = {E3(abs(prod - s['W']['HRP'][iU]))}")
    order = s["info"]["order"]
    pp = h.pos_path(Sigma_np, s["info_pos"]["order"], iU)
    pp_display = [(a, b, round(c, 4)) for a, b, c in pp]
    print("UUP positional path (own-side n, other-side n, share to own side):", pp_display,
          f" isolation depth={len(pp)}")
    print("leaf order (tree):", [cols[j] for j in order])

In [ ]:
for asof in DATES:
    s = snap[asof]
    Sigma_np, iU = s["Sigma_np"], s["iU"]
    rest = [j for j in range(s["n"]) if j != iU]
    V_rest = h.ivp_var_independent(Sigma_np, rest)
    var_uup = float(Sigma_np[iU, iU])
    w_id = V_rest / (var_uup + V_rest)
    w_hrp = float(s["W"]["HRP"][iU])
    w_cf = float(s["w_hrp_cf"][iU])
    rs = h.root_singleton(s["info"]["Z"], iU)
    rs_cf = h.root_singleton(s["info_cf"]["Z"], iU)
    print(f"{asof.date()}: root singleton={rs} (Sigma_cf: {rs_cf})  sigma_UUP={np.sqrt(var_uup):.4f}  "
          f"sqrt(V_rest)={np.sqrt(V_rest):.4f}  r={np.sqrt(V_rest / var_uup):.4f}")
    print(f"    identity={w_id:.4f}  w_UUP(HRP)={w_hrp:.4f}  |diff|={abs(w_hrp - w_id):.3e}  "
          f"w_UUP(HRP, Sigma_cf)={w_cf:.4f}")
    if rs:
        assert abs(w_hrp - w_id) <= 1e-12, (asof, w_hrp, w_id)
    if rs and rs_cf:
        assert abs(w_cf - w_hrp) <= 1e-12, (asof, w_cf, w_hrp)

UUP is a root singleton at all three dates, under both Σ and Σ_cf; its HRP weight equals the identity to 1.7e-16 and does not move when its correlations are zeroed. r is 0.63 and 0.50 at 2014-12-31 and 2017-12-31 but 1.11 at 2022-12-31: the inverse-variance portfolio of the other 12 is then riskier than UUP alone, so UUP gets more than half the book.

### 5.5 Zeroing UUP's correlations

UUP's weight under Σ and under Σ_cf (UUP's correlations set to 0) for HRP, GMV, MDP and ERC. H2b counts the same comparison at every rebalance (§7.3).

In [ ]:
for asof in DATES:
    s = snap[asof]
    iU = s["iU"]
    Z_cf = s["info_cf"]["Z"]
    dcf = {"HRP": abs(s["W"]["HRP"][iU] - s["w_hrp_cf"][iU])}
    dcf.update({m: abs(s["comps"][m][iU] - s["comps_cf"][m][iU]) for m in s["comps"]})
    print(f"\n----- {asof.date()} -----  UUP root singleton under Sigma_cf: {h.root_singleton(Z_cf, iU)}")
    cf_tbl = pd.DataFrame({
        "w_UUP": {"HRP": s["W"]["HRP"][iU], **{m: s["comps"][m][iU] for m in s["comps"]}},
        "w_UUP_cf": {"HRP": s["w_hrp_cf"][iU], **{m: s["comps_cf"][m][iU] for m in s["comps"]}},
        "abs_delta": dcf,
    })
    print(cf_tbl.to_string(float_format=F4))

Zeroing UUP's correlations leaves HRP's UUP weight unchanged at all three dates and cuts GMV's by 11.1–13.1 points, MDP's by 11.8–14.9 and ERC's by 8.8–22.5. GMV, MDP and ERC hold UUP partly for its negative correlations; HRP, with UUP a root singleton, does not.

### 5.6 Recipe sensitivity

Half-L1 distance from registered HRP under average or ward linkage, a direct distance (no distance of distances), positional bisection, and $d = \sqrt{1-\rho}$, which rescales every distance by $\sqrt{2}$ and should not move the weights.

In [ ]:
for asof in DATES:
    s = snap[asof]
    w_hrp = s["W"]["HRP"]
    rec = {"alt_" + k: ml.diagnostics.half_l1(v, w_hrp) for k, v in s["alts"].items()}
    rec["sqrt(1-rho)_sanity"] = ml.diagnostics.half_l1(s["w_sanity"], w_hrp)
    rec["tree_vs_positional"] = ml.diagnostics.half_l1(s["W"]["HRP[pos]"], w_hrp)
    print(f"\n----- {asof.date()} (half-L1 vs registered HRP) -----")
    print(pd.Series(rec).to_string(float_format=E3))

Over the 3 dates the largest shifts are ward 17.4%, average 9.2% and direct 7.4% half-L1 (the 7–17% range in §2), and positional bisection moves the weights by 12.5–19.9%; $\sqrt{1-\rho}$ moves them by at most 1.8e−16. The recipe is not robust to the linkage choice; ward is the backtested linkage sensitivity (§2).

### 5.7 Snapshot map

The methods in (volatility, μ) space on the 2022-12-31 sample Σ. MaxSharpe is computed at this date only; it is not backtested.

In [ ]:
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()

s2022 = snap[asof_2022]
w_hrp_2022 = s2022["W"]["HRP"]
w_gmv_2022 = s2022["W"]["GMV"]
w_mdp_2022 = s2022["W"]["MDP"]
w_erc_2022 = s2022["W"]["ERC"]
w_ivp_2022 = s2022["W"]["IVP"]
w_ew_2022 = s2022["W"]["EW"]

rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
if excess_2022.max() > 0:
    w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "F1", asof_2022)
else:
    w_msr_2022 = _min_variance_long_only(Sigma_2022)

w_6040_2022 = results["60/40"]["wlog"].loc[asof_2022].reindex(Sigma_2022.columns).to_numpy()

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022,
    {"HRP": w_hrp_2022, "GMV": w_gmv_2022, "MDP": w_mdp_2022, "ERC": w_erc_2022,
     "MaxSharpe": w_msr_2022, "EqualWeight": w_ew_2022, "IVP": w_ivp_2022, "60/40": w_6040_2022},
    title=f"Snapshot {asof_2022.date()} -- HRP / GMV / MDP / ERC / MaxSharpe / EW / IVP / 60/40",
    figsize=(10, 6), legend_outside=True,
    styles={"60/40": {"marker": "D", "s": 130, "color": ml.plotting.METHOD_COLORS["60/40"], "label": "60/40"}},
)
plt.show()

print(f"w_UUP: HRP={w_hrp_2022[s2022['iU']]:.4f}  GMV={w_gmv_2022[s2022['iU']]:.4f}  "
      f"MDP={w_mdp_2022[s2022['iU']]:.4f}  ERC={w_erc_2022[s2022['iU']]:.4f}  "
      f"MaxSharpe={w_msr_2022[s2022['iU']]:.4f}  EW={w_ew_2022[s2022['iU']]:.4f}  "
      f"IVP={w_ivp_2022[s2022['iU']]:.4f}")

HRP sits just right of GMV, MDP and ERC at the low-volatility end (ex-ante 4.59% against 3.32%, 3.50% and 4.32%), with IVP further right at 7.56%. Its UUP weight, 55.30%, is between ERC's 48.36% and GMV's 58.12%.

### 5.8 Mechanism grid

One row per date. (a) Dendrogram: each node's share to its left &#124; right branch and each leaf's final HRP weight. (b) Correlations in leaf order, UUP boxed, dashed line at the root split. (c) UUP's weight by method; hollow bars under Σ_cf.

In [ ]:
fig, annotated_counts = h.mechanism_grid(snap, DATES)
assert all(annotated == n_links for _, annotated, n_links in annotated_counts)
plt.show()

At every date UUP splits off alone at the root, the last merge at heights 1.08, 1.10 and 1.35, and the root share to UUP's side is 0.281, 0.201 and 0.553. In (c) the hollow Σ_cf bar matches the solid one for HRP only.

## §6 Out-of-sample results

### 6.1 Summary

Walk-forward as in notebook 01 §5; the test window (2023+) is the cleaner read, and wealth paths and formal tests are in notebook 08.

In [ ]:
cols = ["ann_return", "ann_vol", "sharpe", "max_dd", "ann_turnover"]
group_order = ["registered", "sensitivity", "reference", "reproduction gate", "benchmark"]
window_order = ["full", "train (<= split)", "test (> split)"]
summary_table2 = summary_table[cols].reset_index()
summary_table2["group"] = pd.Categorical(summary_table2["strategy"].map(h.run_group), group_order, ordered=True)
assert summary_table2["group"].notna().all()
summary_table2["strategy"] = pd.Categorical(summary_table2["strategy"], list(results), ordered=True)
summary_table2["window"] = pd.Categorical(summary_table2["window"], window_order, ordered=True)
summary_table2 = summary_table2.set_index(["group", "strategy", "window"]).sort_index()
print(summary_table2.round(4).to_string())

HRP(S) runs at 3.88% volatility, above GMV(S)'s 3.17% (ERC(S) 3.94%, EW 7.87%), the reverse of the lower out-of-sample variance López de Prado (2016) reports for HRP in simulation. Its full-window Sharpe ratio is 0.64 (train 0.60, test 0.82), below EW's 0.74 and 60/40's 0.91 (test 1.00); the three sensitivities run 0.64–0.69. Turnover is 61%/yr (GMV(S) 71%, ERC(S) 27%).

### 6.2 Paired differences (H4–H6)

For the eight registered pairs: daily return differences, each run's CAPM α and β (60/40 as context), the regression of each difference on the market (Δα with its own t), and the β split total = β part + Δα; plain OLS, no HAC (notebook 08 uses Newey–West).

In [ ]:
pairs = [
    ("HRP(S)", "GMV(S)"),
    ("HRP(S)", "MDP(S)"),
    ("HRP(S)", "ERC(S)"),
    ("HRP(S)", "EW"),
    ("HRP(LW)", "HRP(S)"),
    ("HRP(S)", "IVP"),
    ("HRP[pos](S)", "HRP(S)"),
    ("HRP[ward](S)", "HRP(S)"),
]

paired_table, paired_series = ml.inference.paired_table(results, pairs, split_ts, summary_table)
print(paired_table[["ann_mean_diff", "TE", "t", "cost_drag_diff"]].to_string(float_format=F4))

HRP(S) − GMV(S) is +0.52%/yr (t 1.25) and HRP(S) − ERC(S) −0.52%/yr (t −1.14); HRP(S) − EW is −3.30%/yr (t −1.94) and −3.57%/yr in the test window. No total difference has a t beyond ±2, and cost-drag differences are at most 0.05%/yr.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, list(strategies), MKT_simple, rf_daily)
print(capm_table.to_string(float_format=F4))

In [ ]:
diff_capm_table = ml.inference.diff_regression(paired_series, pairs, MKT_simple)
print(diff_capm_table.to_string(float_format=F4))

In [ ]:
decomp_results = ml.inference.beta_decomposition(capm_table, paired_table, diff_capm_table, pairs,
                                                 results, rf_daily, MKT_simple)
print(ml.inference.format_beta_decomposition(decomp_results, capm_table, diff_capm_table), end="")

β split (full window): HRP(S) − GMV(S) is Δα +0.63%/yr (t 1.53) against a β part of −0.12%/yr; HRP(S) − EW is β part −4.60%/yr against Δα +1.30%/yr (t 1.23), mostly β. The other Δα t are 0.54 against MDP(S), 1.59 against ERC(S), 1.01 against IVP, and −0.33 and −0.51 for positional and ward; HRP(LW) − HRP(S) has total t 0.27 full and −0.31 test. HRP(S)'s own CAPM α has t 2.02 on plain OLS; notebook 08 finds no α significant after Newey–West and Romano–Wolf.

Cumulative paired daily differences. Top: β-adjusted (solid, $d_t - \hat\beta\,\mathrm{mkt}_t$) and total (dashed) for the pairs registered on Δα (H4, H6). Bottom: the pairs registered on the total difference (H5, H6 LW).

In [ ]:
alpha_pairs = [
    ("HRP(S)", "GMV(S)"), ("HRP(S)", "MDP(S)"), ("HRP(S)", "ERC(S)"),
    ("HRP(S)", "IVP"), ("HRP[pos](S)", "HRP(S)"), ("HRP[ward](S)", "HRP(S)"),
]
other_pairs = [("HRP(S)", "EW"), ("HRP(LW)", "HRP(S)")]

fig, axes = plt.subplots(2, 1, figsize=(9.5, 9), sharex=True)
ml.plotting.cumulative_paired(
    paired_series, alpha_pairs, split_ts,
    beta={f"{a} - {b}": diff_capm_table.loc[f"{a} - {b}", "beta_diff"] for a, b in alpha_pairs},
    mkt_excess=MKT_simple, ax=axes[0],
    title="(a) Delta-alpha registered pairs: solid = beta-adjusted, dashed = total",
)
ml.plotting.cumulative_paired(
    paired_series, other_pairs, split_ts, ax=axes[1],
    title="(b) HRP-EW and HRP(LW)-HRP(S): total", ylabel="cumulative diff (%)", legend_kw={"fontsize": 8},
)
plt.tight_layout(); plt.show()

The endpoints match 17.2 years × the annualized means: HRP(S) − EW ends near −56.8% and HRP(LW) − HRP(S) near +0.5%; in (a) the β-adjusted lines end between −2.2% (ward) and +11.1% (IVP), the totals between −9.0% (ERC) and +8.9% (GMV). HRP − IVP's β-adjusted line is flat to negative until 2022 and rises after; HRP − EW loses most in 2009 and 2020–21.

### 6.3 Forecast bias (H7)

For each holding segment, q = log(realized volatility / ex-ante $\sqrt{w'\Sigma w}$), with the cost-booking day excluded and segments under 10 days dropped. H7 is the mean of q_HRP − q_GMV; the other methods are descriptive.

In [ ]:
h7_methods = ["HRP(S)", "GMV(S)", "MDP(S)", "ERC(S)", "IVP", "EW"]

q_table, n_dropped, sigma_cache = h.forecast_bias_table(panel, rdates, results, simple_returns, h7_methods)
print(f"segments dropped (< 10 remaining days after excluding cost-booking day): {n_dropped}")

In [ ]:
q_pivot = q_table.pivot(index="asof", columns="method", values="q")
test_mask_q = q_pivot.index > split_ts

print("mean q by method (full / test):")
q_means = pd.DataFrame({
    "full": q_pivot.mean(),
    "test": q_pivot.loc[test_mask_q].mean(),
})
print(q_means.to_string(float_format=F4))

d_h7 = (q_pivot["HRP(S)"] - q_pivot["GMV(S)"]).dropna()
d_full = d_h7
d_test = d_h7.loc[d_h7.index > split_ts]

t_full = float(d_full.mean() / (d_full.std(ddof=1) / np.sqrt(len(d_full))))
t_test = float(d_test.mean() / (d_test.std(ddof=1) / np.sqrt(len(d_test)))) if len(d_test) > 1 else float("nan")

if t_full <= -2 and float(d_test.mean()) < 0:
    h7_verdict = "SUPPORTED"
elif t_full <= -2:
    h7_verdict = "NOT ROBUST"
else:
    h7_verdict = "NOT SUPPORTED"

print(f"\nH7: d = q_HRP - q_GMV  full mean={float(d_full.mean()):.4f} t={t_full:.2f} n={len(d_full)}   "
      f"test mean={float(d_test.mean()):.4f} t={t_test:.2f} n={len(d_test)}  -> {h7_verdict}")

q_HRP and q_GMV per holding segment.

In [ ]:
fig = h.forecast_bias_figure(q_pivot, split_ts)
plt.show()

Every method's mean q is negative (full window from GMV(S)'s −0.0356 to EW's −0.1885). H7 holds as registered (mean q_HRP − q_GMV −0.0643, t −6.18; test −0.0788), but notebook 09 §6b writes q̄ = ½·log v̄ − J and splits the difference into +0.010 from ½·log v̄ and −0.074 from the Jensen gap J ≥ 0, which pulls q̄ below 0 even for a correct forecast. HRP's q is lower because its log ratio is more dispersed, not because its variance forecast is more conservative; that part goes the other way.

## §7 What drove the results

### 7.1 Weights by asset group

Capital by asset group over time for HRP(S), ERC(S), MDP(S) and GMV(S). Below is the mean over the 208 rebalances, with EW and 60/40 for reference (%).

In [ ]:
fig = ml.plotting.group_stackplot(results, ["HRP(S)", "ERC(S)", "MDP(S)", "GMV(S)"], split_ts=split_ts)
plt.show()

group_means = {}
for name in ["HRP(S)", "ERC(S)", "MDP(S)", "GMV(S)", "EW", "60/40"]:
    g = ml.diagnostics.weights_by_group(results[name]["wlog"])
    assert len(g) == 208, (name, len(g))
    group_means[name] = g.mean()
group_means = pd.DataFrame(group_means)
print((100 * group_means).to_string(float_format=lambda v: f"{v:.1f}"))

On average HRP(S) holds 32.6% in fx (UUP) and 2.0% in equity; ERC(S) holds 32.0% and 10.1%, MDP(S) 44.8% and 13.6%, GMV(S) 37.0% and 7.0%. HRP(S) is the most credit-heavy of the four (29.9%) and holds the least rates (8.1%).

### 7.2 The UUP corner (H1a, H1b)

UUP target weight under HRP(S) against GMV(S) and MDP(S) jointly (H1a) and against ERC(S) (H1b, descriptive), at the 208 rebalances (40 in test).

In [ ]:
w_uup_hrp_ts = results["HRP(S)"]["wlog"]["UUP"]
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]
w_uup_erc_ts = results["ERC(S)"]["wlog"]["UUP"]

lt_gmv = w_uup_hrp_ts < w_uup_gmv_ts
lt_mdp = w_uup_hrp_ts < w_uup_mdp_ts
h1a_full = lt_gmv & lt_mdp
n_full = len(h1a_full)
test_mask = h1a_full.index > split_ts
h1a_test = h1a_full.loc[test_mask]
n_test = int(test_mask.sum())
frac_full_h1a = float(h1a_full.mean())
frac_test_h1a = float(h1a_test.mean())

uup_2022 = {
    "HRP(S)": float(w_uup_hrp_ts.loc[asof_2022]),
    "GMV(S)": float(w_uup_gmv_ts.loc[asof_2022]),
    "MDP(S)": float(w_uup_mdp_ts.loc[asof_2022]),
    "ERC(S)": float(w_uup_erc_ts.loc[asof_2022]),
}
h1a_holds_2022 = uup_2022["HRP(S)"] < uup_2022["GMV(S)"] and uup_2022["HRP(S)"] < uup_2022["MDP(S)"]
h1a_verdict = ml.inference.count_rule(h1a_holds_2022, frac_full_h1a, frac_test_h1a)

print(f"H1a (HRP<GMV and HRP<MDP): full {int(h1a_full.sum())}/{n_full} ({100*frac_full_h1a:.1f}%), "
      f"test {int(h1a_test.sum())}/{n_test} ({100*frac_test_h1a:.1f}%)  -> {h1a_verdict}")
print(f"2022-12-31 w_UUP: HRP={uup_2022['HRP(S)']:.4f}  GMV={uup_2022['GMV(S)']:.4f}  "
      f"MDP={uup_2022['MDP(S)']:.4f}  ERC={uup_2022['ERC(S)']:.4f}")

h1b_full = w_uup_hrp_ts < w_uup_erc_ts
h1b_test = h1b_full.loc[test_mask]
print(f"\nH1b (descriptive, HRP<ERC): full {int(h1b_full.sum())}/{n_full} ({100*h1b_full.mean():.1f}%), "
      f"test {int(h1b_test.sum())}/{n_test} ({100*h1b_test.mean():.1f}%)")

H1a holds at 2022-12-31 and at 159/208 rebalances, but at only 14/40 in the test window: NOT ROBUST. HRP's UUP weight is below ERC's at 105/208 and 3/40 (H1b, descriptive).

### 7.3 Correlation blindness (H2a, H2b)

Root-singleton flags and the absolute change in UUP's weight from Σ to Σ_cf for HRP, GMV, MDP and ERC, recomputed on the sample Σ at every rebalance.

In [ ]:
h2_table = h.h2_counterfactual_table(panel, rdates)

In [ ]:
n_h2a = int(h2_table["root_singleton"].sum())
n_both_singleton = int((h2_table["root_singleton"] & h2_table["root_singleton_cf"]).sum())
print(f"H2a (descriptive): UUP root singleton at {n_h2a}/208 rebalances")
print(f"both-singleton count (Sigma and Sigma_cf): {n_both_singleton}/208")

test_mask_h2 = h2_table.index > split_ts
h2b_verdicts = {}
for comp in ["GMV", "MDP", "ERC"]:
    holds = h2_table["dw_HRP"] < h2_table[f"dw_{comp}"]
    frac_full = float(holds.mean())
    frac_test = float(holds.loc[test_mask_h2].mean())
    holds_2022 = bool(holds.loc[asof_2022]) if asof_2022 in holds.index else bool(holds.iloc[-1])
    verdict = ml.inference.count_rule(holds_2022, frac_full, frac_test)
    h2b_verdicts[comp] = dict(frac_full=frac_full, frac_test=frac_test, holds_2022=holds_2022, verdict=verdict)
    print(f"H2b vs {comp}: full {int(holds.sum())}/208 ({100*frac_full:.1f}%), "
          f"test {int(holds.loc[test_mask_h2].sum())}/{int(test_mask_h2.sum())} ({100*frac_test:.1f}%), "
          f"2022-12-31 holds={holds_2022}  -> {verdict}")

UUP is a root singleton at 183/208 rebalances, and under both Σ and Σ_cf at 167/208; at those, HRP's change is 0 by the §3 identity. H2b is supported against GMV, MDP and ERC, and those 167 rebalances are most of its content.

### 7.4 Concentration and turnover (H3)

Effective N of HRP(S) against GMV(S), MDP(S) and ERC(S) at the 208 rebalances; then effective N, largest weight, UUP weight and trailing 12-rebalance turnover over time, with 60/40 as a line and EW dashed.

In [ ]:
effN_hrp_ts = 1.0 / (results["HRP(S)"]["wlog"] ** 2).sum(axis=1)
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)
effN_erc_ts = 1.0 / (results["ERC(S)"]["wlog"] ** 2).sum(axis=1)

h3_specs = [
    ("GMV", effN_gmv_ts, lambda h, c: h > c),
    ("MDP", effN_mdp_ts, lambda h, c: h > c),
    ("ERC", effN_erc_ts, lambda h, c: h < c),
]
h3_verdicts = {}
for comp, comp_ts, op in h3_specs:
    holds = op(effN_hrp_ts, comp_ts)
    frac_full = float(holds.mean())
    frac_test = float(holds.loc[test_mask].mean())
    holds_2022 = bool(holds.loc[asof_2022])
    verdict = ml.inference.count_rule(holds_2022, frac_full, frac_test)
    h3_verdicts[comp] = dict(frac_full=frac_full, frac_test=frac_test, holds_2022=holds_2022, verdict=verdict)
    direction = ">" if op(1, 0) else "<"
    print(f"H3 effN(HRP) {direction} effN({comp}): full {int(holds.sum())}/208 ({100*frac_full:.1f}%), "
          f"test {int(holds.loc[test_mask].sum())}/{int(test_mask.sum())} ({100*frac_test:.1f}%), "
          f"2022-12-31 holds={holds_2022}  -> {verdict}")

print(f"\nmedian effN: HRP(S)={float(effN_hrp_ts.median()):.4f}  GMV(S)={float(effN_gmv_ts.median()):.4f}  "
      f"MDP(S)={float(effN_mdp_ts.median()):.4f}  ERC(S)={float(effN_erc_ts.median()):.4f}")
print(f"2022-12-31 effN: HRP={float(effN_hrp_ts.loc[asof_2022]):.4f}  GMV={float(effN_gmv_ts.loc[asof_2022]):.4f}  "
      f"MDP={float(effN_mdp_ts.loc[asof_2022]):.4f}  ERC={float(effN_erc_ts.loc[asof_2022]):.4f}")

In [ ]:
fig = ml.plotting.concentration_panel(
    results, ["HRP(S)", "ERC(S)", "MDP(S)", "GMV(S)", "60/40"], split_ts=split_ts,
    rows=("effN", "max_w", "highlight", "turnover"), figsize=(9.5, 14),
    suptitle="HRP(S) vs ERC(S) vs MDP(S) vs GMV(S), 60/40 reference: effN, max weight, UUP weight, turnover",
)
plt.show()

Median effective N is 5.03 for HRP(S), between GMV(S)'s 3.79 and ERC(S)'s 6.86 (MDP(S) 3.35); H3 holds against all three. At 2022-12-31 HRP's effective N is 2.93. Its trailing turnover swings between about 30% and 110%/yr, in GMV(S)'s range and far above ERC(S)'s 20–40%.

### 7.5 The root split over time

At a root singleton UUP's weight is one comparison, σ²_UUP against V_rest (§3). The panels show σ_UUP and √V_rest, HRP's UUP weight against the identity, and ρ̄_rest (the mean correlation among the other 12); rebalances where UUP is not a root singleton are shaded. Descriptive.

In [ ]:
f7_table = h.root_split_table(sigma_cache, h2_table, results, rdates)

root_mask = f7_table["root_singleton"]
max_dev = float((f7_table.loc[root_mask, "w_UUP_HRP"] - f7_table.loc[root_mask, "identity"]).abs().max())
assert max_dev <= 1e-12, f"root-singleton identity check failed: max_dev={max_dev:.3e}"
print(f"max |w_UUP(HRP) - identity| over root-singleton rebalances: {max_dev:.3e}")

fig = h.root_split_figure(f7_table, split_ts)
plt.show()

train_mask_f7 = f7_table.index <= split_ts
test_mask_f7 = f7_table.index > split_ts

f7_summary = pd.DataFrame({
    "train (<= split)": h.f7_summary_row(f7_table, train_mask_f7),
    "test (> split)": h.f7_summary_row(f7_table, test_mask_f7),
    "2022-12-31": h.f7_summary_row(f7_table, pd.Timestamp("2022-12-31")),
}).T
print(f7_summary.to_string(float_format=lambda v: f"{v:.4f}"))

From train to test the median σ_UUP moves from 0.0752 to 0.0741, while the median √V_rest rises from 0.0448 to 0.0634 and the median ρ̄_rest from 0.2409 to 0.4440. The median ratio √V_rest/σ_UUP crosses 1 (0.5846 → 1.0304), and HRP's median UUP weight goes from 0.2584 to 0.5149; UUP is a root singleton at 85.12% of train and 100.00% of test rebalances. The corner comes back through the variance of the rest of the book; at a root singleton UUP's correlations do not enter its weight.

## Verdict table

The §2 rules applied to the statistics above: H1a in §7.2, H2b in §7.3, H3 in §7.4, H4–H6 in §6.2, H7 in §6.3. H1b and H2a are descriptive and printed below the table.

In [ ]:
reg = ml.inference.load_registration("nb07")

d5 = decomp_results["HRP(S) - EW"]
t_full_ew = float(paired_table.loc[("HRP(S) - EW", "full"), "t"])
mean_test_ew = float(paired_table.loc[("HRP(S) - EW", "test (> split)"), "ann_mean_diff"])
t_full_lw = float(paired_table.loc[("HRP(LW) - HRP(S)", "full"), "t"])
t_test_lw = float(paired_table.loc[("HRP(LW) - HRP(S)", "test (> split)"), "t"])

stats = {
    "H1a": dict(holds_2022=h1a_holds_2022, frac_full=frac_full_h1a, frac_test=frac_test_h1a,
                display=f"full {int(h1a_full.sum())}/{n_full}, test {int(h1a_test.sum())}/{n_test}, "
                        f"2022-12-31 holds={h1a_holds_2022}"),
}
for comp in ["GMV", "MDP", "ERC"]:
    v = h2b_verdicts[comp]
    stats[f"H2b_{comp}"] = dict(holds_2022=v["holds_2022"], frac_full=v["frac_full"], frac_test=v["frac_test"],
                                display=f"full {100*v['frac_full']:.1f}%, test {100*v['frac_test']:.1f}%, 2022 holds={v['holds_2022']}")
for comp in ["GMV", "MDP", "ERC"]:
    v = h3_verdicts[comp]
    stats[f"H3_{comp}"] = dict(holds_2022=v["holds_2022"], frac_full=v["frac_full"], frac_test=v["frac_test"],
                               display=f"full {100*v['frac_full']:.1f}%, test {100*v['frac_test']:.1f}%, 2022 holds={v['holds_2022']}")
for hid, pair_key in [
    ("H4_GMV", "HRP(S) - GMV(S)"), ("H4_MDP", "HRP(S) - MDP(S)"), ("H4_ERC", "HRP(S) - ERC(S)"),
    ("H6_IVP", "HRP(S) - IVP"), ("H6_pos", "HRP[pos](S) - HRP(S)"), ("H6_ward", "HRP[ward](S) - HRP(S)"),
]:
    t = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])
    stats[hid] = dict(t=t, display=f"t={t:.2f}")
stats["H5"] = dict(t_full=t_full_ew, mean_test=mean_test_ew, beta_part=d5["beta_part"], d_alpha=d5["d_alpha"],
                   display=f"t_full={t_full_ew:.2f}, mean_test={mean_test_ew:.4f}, "
                           f"|beta_part|={abs(d5['beta_part']):.4f} vs |d_alpha|={abs(d5['d_alpha']):.4f}")
stats["H6_LW"] = dict(t_full=t_full_lw, t_test=t_test_lw, display=f"t_full={t_full_lw:.2f}, t_test={t_test_lw:.2f}")
stats["H7"] = dict(t_full=t_full, mean_test=float(d_test.mean()),
                   display=f"t_full={t_full:.2f}, mean_test={float(d_test.mean()):.4f}")

verdict_table = ml.inference.evaluate(reg, stats)
print(verdict_table.to_string(index=False))

print("\n--- descriptive rows ---")
print(f"H1b (descriptive, HRP<ERC): full {int(h1b_full.sum())}/{n_full}, test {int(h1b_test.sum())}/{n_test}")
print(f"H2a (descriptive, UUP root singleton): {n_h2a}/208")
print(f"both-singleton count (Sigma and Sigma_cf): {n_both_singleton}/208")

## §8 TAKEAWAY — the corner comes back through V_rest, not through UUP's own correlations

- **H1a NOT ROBUST.** HRP's UUP weight is below GMV's and MDP's at 159/208 rebalances but at 14/40 in the test window. At 2022-12-31 it is 55.30% (GMV 58.12%, MDP 59.22%, ERC 48.36%).
- **H2b SUPPORTED against all three, mostly by construction.** UUP is a root singleton at 183/208 rebalances, 167 of them under both Σ and Σ_cf; there its weight is V_rest/(σ²_UUP + V_rest) and does not depend on its correlations.
- **H3 SUPPORTED against all three.** Median effective N is 5.03, between GMV's 3.79 and ERC's 6.86 (MDP 3.35).
- **No return difference passes a registered test.** The Δα nulls hold against GMV (t 1.53, narrowly), MDP (0.54) and ERC (1.59, narrowly); H5 is not supported (t −1.94); the H6 sensitivity nulls hold.
- **H7 SUPPORTED, but the difference is the Jensen gap.** Mean q_HRP − q_GMV is −0.0643 (t −6.18); notebook 09 splits it into +0.010 from ½·log v̄ and −0.074 from J.

HRP makes UUP's weight blind to its correlations; after 2022 the corner returns through one variance comparison, as the median √V_rest/σ_UUP crosses 1 (0.5846 → 1.0304).

## Notes

- HAP/HARP-style hierarchical allocators need an expected-return signal at each split, so they are return-based rather than risk-based and are not backtested here.
- Hierarchical clustering portfolios (Raffinot 2017) follow the tree down to a cluster cut and give each cluster equal capital; they share the tree-following step with registered HRP, and the literature reports them as sensitive to the number of clusters.
- The positional (equal-halves) bisection is the original algorithm (López de Prado 2016); it is carried only as a sensitivity because its weights depend on the input column order (§3, §4).